# Práctica 09 — Topic modeling

**Curso:** NLP Clásico con Python · **Clase 09** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase09_topic_modeling.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios 1 a 5 usan un mini-corpus de 8 frases (fútbol y cocina) y cifras que puedes calcular a mano. El reto usa el extracto del Quijote de `datos/gutenberg/` (en Colab, sube la carpeta `datos/`): un texto literario antiguo, no el español de hoy (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `scikit-learn`, `gensim` y `nltk`.

---
## Celda 0 — Setup y datos

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("gensim", "gensim")
asegurar("nltk", "nltk")
asegurar("spacy", "spacy")
print("Librerías listas.")

In [ ]:
import os
import re
import math
import random
import subprocess
import sys
from collections import Counter

import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

import warnings
warnings.filterwarnings("ignore")
random.seed(42)                          # semilla fija
np.random.seed(42)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def cargar_stopwords(idioma):
    """Lista de stopwords de NLTK (la descarga si falta)."""
    try:
        return set(stopwords.words(idioma))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        return set(stopwords.words(idioma))

ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es
CORPORA = {"spa": ("gutenberg/quijote_cap01-14.txt", "spanish"),
           "ita": ("gutenberg/promessi_sposi_cap01-06.txt", "italian")}

def leer_texto(ruta):
    """Lee un extracto: quita las marcas _cursiva_ y los encabezados de capítulo y devuelve un solo texto."""
    parrafos = [p.strip() for p in open(ruta, encoding="utf-8").read().replace("_", "").split("\n\n") if p.strip()]
    return " ".join(p for p in parrafos if not ENCAB.match(p))

def nombres_propios(texto, minimo=3, umbral=0.8):
    """Heurística: palabras que van con mayúscula al menos el 80 % de las veces que aparecen fuera del inicio de frase."""
    con_mayus, total = Counter(), Counter()
    for m in re.finditer(r"\w+", texto):
        if re.search(r"(^|[.!?…:]\s*[«—\"“]?\s*)$", texto[max(0, m.start() - 4):m.start()]):
            continue                                   # inicio de frase: la mayúscula no informa
        w = m.group()
        total[w.lower()] += 1
        con_mayus[w.lower()] += w[0].isupper()
    return {w for w in total if total[w] >= minimo and con_mayus[w] / total[w] >= umbral}

def trozos(tokens, n=300):
    """Parte una lista de tokens en pseudodocumentos consecutivos de n tokens (el último trozo corto se descarta si tiene menos de n/2)."""
    return [tokens[i:i + n] for i in range(0, len(tokens) - n // 2, n)]

def temas_lda(m, n=8):
    """Las n palabras más probables de cada tema."""
    return [[w for w, _ in m.show_topic(i, n)] for i in range(m.num_topics)]

def coherencia(m, docs, dic, bow, medida="c_v"):
    """Coherencia de gensim (c_v usa una ventana deslizante sobre los textos; u_mass usa los recuentos)."""
    if medida == "u_mass":
        return CoherenceModel(model=m, corpus=bow, dictionary=dic, coherence="u_mass").get_coherence()
    return CoherenceModel(model=m, texts=docs, dictionary=dic, coherence=medida, processes=1).get_coherence()

print("Funciones auxiliares cargadas.")

In [ ]:
# datos-es
# Dos temas inventados sobre un vocabulario de 4 palabras (una fila por tema; cada fila suma 1)
VOCAB = ["gol", "liga", "sal", "receta"]      # en italiano: gol, campionato, sale, ricetta
PHI = np.array([[0.5, 0.3, 0.1, 0.1],          # tema 1 (P(palabra | tema 1))
                [0.1, 0.1, 0.4, 0.4]])         # tema 2

# Mini-corpus de 8 frases: 4 de fútbol y 4 de cocina (español e italiano)
FRASES = {
    "spa": ["el equipo marcó un gol en la liga", "la liga termina y el equipo gana",
            "el delantero marcó otro gol", "el entrenador cambió al delantero del equipo",
            "la receta lleva sal y aceite", "añade sal a la receta del guiso",
            "el guiso lleva aceite y cebolla", "la cebolla y el aceite del guiso"],
    "ita": ["la squadra ha segnato un gol in campionato", "il campionato finisce e la squadra vince",
            "l'attaccante ha segnato un altro gol", "l'allenatore ha cambiato l'attaccante della squadra",
            "la ricetta richiede sale e olio", "aggiungi sale alla ricetta dello stufato",
            "lo stufato richiede olio e cipolla", "la cipolla e l'olio dello stufato"],
}
TEMA_REAL = ["futbol"] * 4 + ["cocina"] * 4
STOP_E = {"spa": {"el", "la", "un", "y", "en", "del", "al", "otro", "a", "lleva", "añade"},
          "ita": {"il", "la", "lo", "un", "e", "in", "ha", "della", "dello", "alla", "altro", "l", "richiede", "aggiungi"}}

In [ ]:
# Matriz TF-IDF del mini-corpus (español) y SVD truncada de 2 dimensiones, dadas para el ejercicio 3
_docs = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_tf = TfidfVectorizer(analyzer=lambda d: d)
_X = _tf.fit_transform(_docs)
VOCAB_E = [str(w) for w in _tf.get_feature_names_out()]
svd = TruncatedSVD(n_components=2, random_state=0).fit(_X)
print("TF-IDF:", _X.shape, "| valores singulares:", np.round(svd.singular_values_, 3))

## Ejercicio 1 — Mezcla de temas: `P(w | d) = Σ θ · P(w | t)`

Completa `p_palabras(theta, phi)`: dado el vector de proporciones de temas `theta` (suma 1) y la matriz `phi` (una fila por tema, una columna por palabra), devuelve el vector `P(w | d)` de cada palabra del vocabulario. Completa también `tema_dominante(theta)`: el índice del tema con mayor proporción.

Antes de programar, a mano con `PHI` y `θ = (0,2 ; 0,8)`: `P(gol | d) = 0,2·0,5 + 0,8·0,1 = 0,18`.

In [ ]:
def p_palabras(theta, phi):
    return None    # TODO: producto de theta por phi (una línea con NumPy)

def tema_dominante(theta):
    return None    # TODO: índice del máximo

# Comprobación automática
_p = p_palabras([0.2, 0.8], PHI)
assert _p is not None and tema_dominante([0.2, 0.8]) is not None, "Completa las líneas marcadas con TODO"
assert np.allclose(_p, [0.18, 0.14, 0.34, 0.34]), "revisa el producto: P(gol | d) debe ser 0,18"
assert np.isclose(_p.sum(), 1.0) and tema_dominante([0.2, 0.8]) == 1 and tema_dominante([0.6, 0.4]) == 0
assert np.allclose(p_palabras([1, 0], PHI), PHI[0])          # un documento 100 % del tema 1 reproduce el tema
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Pre-tratamiento: stopwords, nombres propios y lemas

Completa `preparar_p(tokens, stop, nombres, lemas=None, minimo=3)`: pasa a minúsculas, y quita las stopwords (`stop`), las palabras de menos de `minimo` letras y los nombres propios (`nombres`). Si se da `lemas` (una lista del mismo tamaño que `tokens`), se usan **los lemas en lugar de las formas**.

Antes de programar, a mano: con `["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]`, `stop = {"dijo", "que", "era"}` y `nombres = {"sancho"}` queda `["cura", "barbero"]` (`el` tiene 2 letras).

In [ ]:
def preparar_p(tokens, stop, nombres, lemas=None, minimo=3):
    fuente = None      # TODO: los lemas si se dan; si no, los tokens en minúsculas
    return None        # TODO: lista de palabras de `fuente` que no estén en stop ni en nombres y tengan al menos `minimo` letras

# Comprobación automática
FRASE_E = ["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]          # dato-es
STOP_2, NOM_2 = {"dijo", "que", "era"}, {"sancho"}                                  # dato-es
LEM_E = ["el", "cura", "decir", "que", "el", "barbero", "ser", "sancho"]            # dato-es
SIN_FILTRO_E = ["el", "cura", "dijo", "que", "el", "barbero", "era", "sancho"]            # dato-es
_r = preparar_p(FRASE_E, STOP_2, NOM_2)
assert _r is not None, "Completa las líneas marcadas con TODO"
assert _r == ["cura", "barbero"], "revisa: minúsculas, stopwords, nombres y palabras cortas"
assert preparar_p(FRASE_E, set(), set(), minimo=1) == SIN_FILTRO_E
assert preparar_p(FRASE_E, STOP_2, NOM_2, lemas=LEM_E) == ["cura", "decir", "barbero", "ser"]
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — LSA: las dimensiones y el signo arbitrario

En la celda de datos se construye una matriz TF-IDF del mini-corpus y se ajusta `TruncatedSVD` con 2 dimensiones (`svd`). Completa `fijar_signo(v)`: devuelve `v` si su componente de mayor valor absoluto es positiva, y `-v` en caso contrario (así dos ajustes que difieren solo en el signo dan el mismo vector). Después completa `palabras_positivas(comp, vocab, n)`: las `n` palabras de mayor peso **tras fijar el signo**.

Comprobación a mano: `fijar_signo([0,1; -0,9; 0,3])` = `[-0,1; 0,9; -0,3]`.

In [ ]:
def fijar_signo(v):
    v = np.asarray(v, dtype=float)
    return None    # TODO: v o -v según el signo de la componente de mayor valor absoluto

def palabras_positivas(comp, vocab, n=3):
    comp = fijar_signo(comp)
    return None    # TODO: las n palabras de mayor peso (usa np.argsort)

# Comprobación automática
assert fijar_signo([0.1, -0.9, 0.3]) is not None, "Completa las líneas marcadas con TODO"
assert np.allclose(fijar_signo([0.1, -0.9, 0.3]), [-0.1, 0.9, -0.3]) and np.allclose(fijar_signo([-0.1, 0.9, -0.3]), [-0.1, 0.9, -0.3])
assert all(np.allclose(fijar_signo(c), fijar_signo(-c)) for c in svd.components_), "fijar_signo(v) y fijar_signo(-v) deben coincidir"
_top = palabras_positivas(svd.components_[1], VOCAB_E, 3)
assert _top is not None and len(_top) == 3 and len(set(_top)) == 3 and all(w in VOCAB_E for w in _top)
_c = fijar_signo(svd.components_[1])
assert [VOCAB_E.index(w) for w in _top] == list(np.argsort(-_c)[:3]), "deben ser las 3 de mayor peso, en orden"
print("Ejercicio 3 correcto ✔", _top)

## Ejercicio 4 — LDA sobre recuentos, con semilla fija

Completa `entrenar(docs, k, semilla=0, no_below=1, no_above=1.0)`: `docs` es una lista de listas de palabras. Construye un `Dictionary`, filtra con `filter_extremes(no_below, no_above)`, calcula los **recuentos** de cada documento con `doc2bow` (no TF-IDF) y entrena un `LdaModel` con `num_topics=k`, `random_state=semilla`, `passes=10`, `iterations=50` y `eval_every=None`. Devuelve `(modelo, diccionario, bow)`.

Comprobaciones a mano: sin filtro, el diccionario tiene tantas entradas como palabras distintas, y los recuentos del documento 0 suman su número de palabras.

In [ ]:
def entrenar(docs, k, semilla=0, no_below=1, no_above=1.0):
    dic = None     # TODO: Dictionary(docs) y luego filter_extremes(no_below=no_below, no_above=no_above)
    bow = None     # TODO: [dic.doc2bow(d) for d in docs]
    modelo = None  # TODO: LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return modelo, dic, bow

# Comprobación automática
DOCS_E = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_m, _dic, _bow = entrenar(DOCS_E, 2)
assert _m is not None and _dic is not None, "Completa las líneas marcadas con TODO"
assert len(_dic) == len({w for d in DOCS_E for w in d}), "sin filtro, el diccionario debe tener todas las palabras distintas"
assert sum(c for _, c in _bow[0]) == len(DOCS_E[0]), "los recuentos del documento 0 deben sumar su número de palabras"
assert _m.get_topics().shape == (2, len(_dic)) and np.allclose(_m.get_topics().sum(axis=1), 1.0, atol=1e-5)
_theta = np.array([[p for _, p in _m.get_document_topics(b, minimum_probability=0.0)] for b in _bow])
assert _theta.shape == (8, 2) and np.allclose(_theta.sum(axis=1), 1.0, atol=1e-5)
_m2 = entrenar(DOCS_E, 2)[0]
assert np.allclose(_m.get_topics(), _m2.get_topics()), "con la misma semilla, los temas deben ser idénticos"
assert entrenar(DOCS_E, 2, semilla=1)[0].num_topics == 2
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Medir si un modelo recupera los temas conocidos: pureza y ARI

Completa `pureza(y, grupos)`: para cada grupo, cuenta la clase mayoritaria (con `Counter`) y suma; divide por el total. Después calcula el ARI con `adjusted_rand_score` para el ejemplo de la celda.

A mano: `y = [a,a,a,b,b,b]` y `grupos = [0,0,1,1,1,1]`. El grupo 0 tiene {a,a} → 2; el grupo 1 tiene {a,b,b,b} → 3. Pureza = 5/6. Los números de los grupos son arbitrarios: renombrarlos no cambia la pureza.

In [ ]:
Y_E = ["a", "a", "a", "b", "b", "b"]        # dato-es
G_E = [0, 0, 1, 1, 1, 1]

def pureza(y, grupos):
    total = 0
    for g in sorted(set(grupos)):
        pass       # TODO: suma la frecuencia de la clase mayoritaria entre los documentos del grupo g
    return None    # TODO: total dividido por len(y)

ari_e = None       # TODO: ARI entre Y_E y G_E

# Comprobación automática
assert pureza(Y_E, G_E) is not None and ari_e is not None, "Completa las líneas marcadas con TODO"
assert math.isclose(pureza(Y_E, G_E), 5 / 6), "la pureza de este ejemplo es 5/6"
assert pureza(Y_E, [1, 1, 0, 0, 0, 0]) == pureza(Y_E, G_E), "renombrar los grupos no cambia la pureza"
assert pureza(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0 and pureza(Y_E, [0, 1, 2, 3, 4, 5]) == 1.0      # ojo: un grupo por documento también da pureza 1
assert math.isclose(ari_e, adjusted_rand_score(Y_E, [1, 1, 0, 0, 0, 0])) and adjusted_rand_score(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0
assert 0 < ari_e < 1
print("Ejercicio 5 correcto ✔", round(pureza(Y_E, G_E), 3), round(ari_e, 3))

---
## Reto final — Los nombres propios y el número de temas en el Quijote

Con `datos/gutenberg/quijote_cap01-14.txt` (español del siglo XVII; *ejemplo ilustrativo*): pseudodocumentos de 300 tokens, con **tus** funciones `preparar_p` (ejercicio 2) y `entrenar` (ejercicio 4).

1. Prepara `docs_sin` (sin nombres propios) y `docs_con` (con nombres propios), ambos con las stopwords de NLTK.
2. Completa `nombres_en_top(modelo, nombres, n=8)`: cuántas de las `n` palabras principales de todos los temas son nombres propios.
3. Entrena LDA con `k = 5`, semilla 0, `no_below=3`, `no_above=0.25` en `docs_con` y `docs_sin` y calcula `n_con` y `n_sin`.
4. Para `k` en (3, 4, 5, 6) y semillas (0, 1), calcula la coherencia **c_v** media (con `coherencia(modelo, docs_sin, dic, bow)`) y guarda en `mejor_k` el `k` con mayor media. Recuerda: es una guía ruidosa, no la respuesta.

In [ ]:
ruta, nombre_nltk = CORPORA["spa"]
texto = leer_texto(cargar_datos(ruta))
NOMBRES = nombres_propios(texto)
STOP = cargar_stopwords(nombre_nltk)
trozos_q = trozos(re.findall(r"\w+", texto), 300)
docs_sin = None    # TODO: preparar_p sobre cada trozo, quitando stopwords y nombres
docs_con = None    # TODO: preparar_p sobre cada trozo, quitando solo las stopwords

def nombres_en_top(modelo, nombres, n=8):
    return None    # TODO: cuenta las palabras de temas_lda(modelo, n) que están en nombres

n_con = n_sin = None   # TODO: nombres en el top de LDA (k=5, semilla 0, no_below=3, no_above=0.25) con docs_con y con docs_sin

coh_media = {}
for k in (3, 4, 5, 6):
    pass           # TODO: media de c_v con semillas 0 y 1 sobre docs_sin (mismos filtros); guárdala en coh_media[k]
mejor_k = None     # TODO: el k con mayor coherencia media

# Comprobación automática
assert docs_sin is not None and n_con is not None and mejor_k is not None, "Completa las líneas marcadas con TODO"
assert len(docs_sin) == len(docs_con) > 90 and all(len(a) <= len(b) for a, b in zip(docs_sin, docs_con))
assert n_sin == 0, "sin nombres propios no debería quedar ninguno entre las palabras principales"
assert n_con >= 3, "con nombres propios deberían aparecer varios"
assert mejor_k in (3, 4, 5, 6) and set(coh_media) == {3, 4, 5, 6}
print("Reto final correcto ✔")
print("n_con =", n_con, "| n_sin =", n_sin, "| c_v media =", {k: round(v, 3) for k, v in coh_media.items()}, "| mejor_k =", mejor_k)

**Tus conclusiones (escribe 3-4 frases):** ¿por qué el signo de una dimensión de LSA no significa nada? ¿Por qué LDA se entrena sobre recuentos y no sobre TF-IDF? ¿Qué decisiones de pre-tratamiento cambiaron los temas del Quijote, y por qué no puedes comparar el c_v de dos preparaciones distintas ni el de un corpus español con el de uno italiano? ¿Quién pone la etiqueta a un tema?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def p_palabras(theta, phi):
    return np.asarray(theta) @ np.asarray(phi)

def tema_dominante(theta):
    return int(np.argmax(theta))

# Comprobación automática
_p = p_palabras([0.2, 0.8], PHI)
assert _p is not None and tema_dominante([0.2, 0.8]) is not None, "Completa las líneas marcadas con TODO"
assert np.allclose(_p, [0.18, 0.14, 0.34, 0.34]), "revisa el producto: P(gol | d) debe ser 0,18"
assert np.isclose(_p.sum(), 1.0) and tema_dominante([0.2, 0.8]) == 1 and tema_dominante([0.6, 0.4]) == 0
assert np.allclose(p_palabras([1, 0], PHI), PHI[0])          # un documento 100 % del tema 1 reproduce el tema
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def preparar_p(tokens, stop, nombres, lemas=None, minimo=3):
    fuente = lemas if lemas is not None else [w.lower() for w in tokens]
    return [w for w in fuente if w not in stop and w not in nombres and len(w) >= minimo]

# Comprobación automática
FRASE_E = ["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]          # dato-es
STOP_2, NOM_2 = {"dijo", "que", "era"}, {"sancho"}                                  # dato-es
LEM_E = ["el", "cura", "decir", "que", "el", "barbero", "ser", "sancho"]            # dato-es
SIN_FILTRO_E = ["el", "cura", "dijo", "que", "el", "barbero", "era", "sancho"]            # dato-es
_r = preparar_p(FRASE_E, STOP_2, NOM_2)
assert _r is not None, "Completa las líneas marcadas con TODO"
assert _r == ["cura", "barbero"], "revisa: minúsculas, stopwords, nombres y palabras cortas"
assert preparar_p(FRASE_E, set(), set(), minimo=1) == SIN_FILTRO_E
assert preparar_p(FRASE_E, STOP_2, NOM_2, lemas=LEM_E) == ["cura", "decir", "barbero", "ser"]
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def fijar_signo(v):
    v = np.asarray(v, dtype=float)
    return v if v[np.argmax(np.abs(v))] > 0 else -v

def palabras_positivas(comp, vocab, n=3):
    comp = fijar_signo(comp)
    return [str(vocab[i]) for i in np.argsort(-comp)[:n]]

# Comprobación automática
assert fijar_signo([0.1, -0.9, 0.3]) is not None, "Completa las líneas marcadas con TODO"
assert np.allclose(fijar_signo([0.1, -0.9, 0.3]), [-0.1, 0.9, -0.3]) and np.allclose(fijar_signo([-0.1, 0.9, -0.3]), [-0.1, 0.9, -0.3])
assert all(np.allclose(fijar_signo(c), fijar_signo(-c)) for c in svd.components_), "fijar_signo(v) y fijar_signo(-v) deben coincidir"
_top = palabras_positivas(svd.components_[1], VOCAB_E, 3)
assert _top is not None and len(_top) == 3 and len(set(_top)) == 3 and all(w in VOCAB_E for w in _top)
_c = fijar_signo(svd.components_[1])
assert [VOCAB_E.index(w) for w in _top] == list(np.argsort(-_c)[:3]), "deben ser las 3 de mayor peso, en orden"
print("Ejercicio 3 correcto ✔", _top)

### Solución del ejercicio 4

In [ ]:
def entrenar(docs, k, semilla=0, no_below=1, no_above=1.0):
    dic = Dictionary(docs)
    dic.filter_extremes(no_below=no_below, no_above=no_above)
    bow = [dic.doc2bow(d) for d in docs]
    modelo = LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return modelo, dic, bow

# Comprobación automática
DOCS_E = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_m, _dic, _bow = entrenar(DOCS_E, 2)
assert _m is not None and _dic is not None, "Completa las líneas marcadas con TODO"
assert len(_dic) == len({w for d in DOCS_E for w in d}), "sin filtro, el diccionario debe tener todas las palabras distintas"
assert sum(c for _, c in _bow[0]) == len(DOCS_E[0]), "los recuentos del documento 0 deben sumar su número de palabras"
assert _m.get_topics().shape == (2, len(_dic)) and np.allclose(_m.get_topics().sum(axis=1), 1.0, atol=1e-5)
_theta = np.array([[p for _, p in _m.get_document_topics(b, minimum_probability=0.0)] for b in _bow])
assert _theta.shape == (8, 2) and np.allclose(_theta.sum(axis=1), 1.0, atol=1e-5)
_m2 = entrenar(DOCS_E, 2)[0]
assert np.allclose(_m.get_topics(), _m2.get_topics()), "con la misma semilla, los temas deben ser idénticos"
assert entrenar(DOCS_E, 2, semilla=1)[0].num_topics == 2
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
Y_E = ["a", "a", "a", "b", "b", "b"]        # dato-es
G_E = [0, 0, 1, 1, 1, 1]

def pureza(y, grupos):
    total = 0
    for g in sorted(set(grupos)):
        total += max(Counter(yi for yi, gi in zip(y, grupos) if gi == g).values())
    return total / len(y)

ari_e = adjusted_rand_score(Y_E, G_E)

# Comprobación automática
assert pureza(Y_E, G_E) is not None and ari_e is not None, "Completa las líneas marcadas con TODO"
assert math.isclose(pureza(Y_E, G_E), 5 / 6), "la pureza de este ejemplo es 5/6"
assert pureza(Y_E, [1, 1, 0, 0, 0, 0]) == pureza(Y_E, G_E), "renombrar los grupos no cambia la pureza"
assert pureza(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0 and pureza(Y_E, [0, 1, 2, 3, 4, 5]) == 1.0      # ojo: un grupo por documento también da pureza 1
assert math.isclose(ari_e, adjusted_rand_score(Y_E, [1, 1, 0, 0, 0, 0])) and adjusted_rand_score(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0
assert 0 < ari_e < 1
print("Ejercicio 5 correcto ✔", round(pureza(Y_E, G_E), 3), round(ari_e, 3))

### Solución del reto final

In [ ]:
ruta, nombre_nltk = CORPORA["spa"]
texto = leer_texto(cargar_datos(ruta))
NOMBRES = nombres_propios(texto)
STOP = cargar_stopwords(nombre_nltk)
trozos_q = trozos(re.findall(r"\w+", texto), 300)
docs_sin = [preparar_p(x, STOP, NOMBRES) for x in trozos_q]
docs_con = [preparar_p(x, STOP, set()) for x in trozos_q]

def nombres_en_top(modelo, nombres, n=8):
    return sum(w in nombres for palabras in temas_lda(modelo, n) for w in palabras)

n_con = nombres_en_top(entrenar(docs_con, 5, 0, 3, 0.25)[0], NOMBRES)
n_sin = nombres_en_top(entrenar(docs_sin, 5, 0, 3, 0.25)[0], NOMBRES)

coh_media = {}
for k in (3, 4, 5, 6):
    valores = []
    for s in (0, 1):
        m, dic, bow = entrenar(docs_sin, k, s, 3, 0.25)
        valores.append(coherencia(m, docs_sin, dic, bow, "c_v"))
    coh_media[k] = float(np.mean(valores))
mejor_k = max(coh_media, key=coh_media.get)

# Comprobación automática
assert docs_sin is not None and n_con is not None and mejor_k is not None, "Completa las líneas marcadas con TODO"
assert len(docs_sin) == len(docs_con) > 90 and all(len(a) <= len(b) for a, b in zip(docs_sin, docs_con))
assert n_sin == 0, "sin nombres propios no debería quedar ninguno entre las palabras principales"
assert n_con >= 3, "con nombres propios deberían aparecer varios"
assert mejor_k in (3, 4, 5, 6) and set(coh_media) == {3, 4, 5, 6}
print("Reto final correcto ✔")
print("n_con =", n_con, "| n_sin =", n_sin, "| c_v media =", {k: round(v, 3) for k, v in coh_media.items()}, "| mejor_k =", mejor_k)

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (tema como distribución, LSA y su signo, LDA y los recuentos, coherencia, pureza y ARI)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*